In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
from ingest import load_faq_data, build_index
import numpy as np
from minsearch import VectorSearch
documents = load_faq_data()

from tqdm.auto import tqdm
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
X=np.array(vectors)


vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

  0%|          | 0/27 [00:00<?, ?it/s]

In [4]:

from rag_helper import RAGBase
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
        
    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict,
            
        )
        
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
    course="llm-zoomcamp"
)

In [5]:
vector_assistant.rag("the program has already begun, can I still sign up?")

{'response': "Yes, you can still sign up, but if you want to receive a certificate, you'll need to submit your project while the submissions are still being accepted.",
 'input_tokens': 387}

In [7]:
#with sqlLite search 
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)
vs_index.fit(vectors, documents)

In [16]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector,filter_dict={"course": "llm-zoomcamp"}, num_results=5)
results
#vs_index.close()

In [19]:
from rag_helper import RAGBase
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=client,
    
)

In [20]:
vector_assistant.rag("the program has already begun, can I still sign up?")

{'response': 'You can still sign up for the program.', 'input_tokens': 387}